# Online mixed layer temperature budget: all individual terms (November 2019)

This notebook reproduces the first spatial plot from the "Plots comparing offline and daily budgets" section of `Offline_Online_Budget_Comparison.ipynb`, but:
- Only processes the **online** budget (based on `process_online_budget_year.sub`), restricted to a single month (November 2019) to keep the computation cheap.
- Instead of showing the grouped budget terms (`advection`, `vert_mixing`, `surface_flux`, `sw_pen`), it shows every individual term that makes up these groups (as defined by `bud_var_grps` in `process_online_budget_year.sub`) separately.

The full field (monthly mean) is plotted, not anomalies relative to a climatology.


In [ ]:
# Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from tqdm import tqdm
import string
import os

from dask.distributed import Client
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures/Online_Offline_Comparison')

In [ ]:
# Load workers:
client = Client(n_workers=4)
client


## Settings and paths

Same simulation/output and region as used in `Offline_Online_Budget_Comparison.ipynb`, but restricted to November 2019 only (plus the bracketing snapshots needed to compute the mixed layer temperature tendency).

In [ ]:
# Settings and paths:
output = 366 # 366 = 2019 - contains 3D daily budget diagnostics

base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
base2 = base + 'output%03d/ocean/' % output

reg = [-270, -70, -70, 60] # Pacific - region used for the comparison figure in Offline_Online_Budget_Comparison.ipynb

# Only process November 2019:
month_str = '2019-11'
times = slice('2019-11-01','2019-11-30')
times_snap = slice('2019-10-31','2019-11-30') # 1 extra (earlier) day, needed to compute the mlt tendency by finite difference

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}

rho0 = 1035.
Cp = 3992.10322329649


## Load data

Load the grid, daily budget and daily snapshot diagnostics (as in `process_online_budget_year.sub`), but subset in space (`reg`) and, once the time coordinate has been decoded, in time (November 2019 only).

In [ ]:
# Grid file (for masking/plotting land):
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Standard average daily budget diagnostics:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085):
ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget.average_DT.data = ds_day_budget.average_DT*np.timedelta64(1,'D')

# Subselect to November 2019 only (budget) and the bracketing snapshots:
ds_day_budget = ds_day_budget.sel(time=times)
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)


## Define budget terms and correction function

`bud_var_grps` (copied from `process_online_budget_year.sub`) defines which raw diagnostic variables make up each grouped budget term. Rather than summing each group into a single variable, we keep every individual variable listed here separate.

In [ ]:
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}

def compute_corrections(ds_day_budget):

    # P-E+R correction: 
    pme_cor = -ds_day_budget['pme_river_times_temp_in_mld']/rho0
    ds_day_budget['sfc_hflux_pme_in_mld_cor'] = ds_day_budget['sfc_hflux_pme_in_mld'] + pme_cor*rho0*Cp

    # Eta-smoother correction:
    eta_smoother_cor = -ds_day_budget['eta_smoother_times_temp_in_mld']/rho0
    ds_day_budget['temp_eta_smooth_in_mld_cor'] = ds_day_budget['temp_eta_smooth_in_mld'] + eta_smoother_cor*rho0*Cp

    # Advection correction:
    adv_cor1 = (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0
    adv_cor2 = ds_day_budget['s_surf_ent_temp']/rho0
    ds_day_budget['temp_advection_in_mld_cor'] = ds_day_budget['temp_advection_in_mld'] + adv_cor1*rho0*Cp  + adv_cor2*rho0*Cp

    # Total correction for entrainment-by-residual:
    ds_day_budget['temp_tendency_in_mld_cor'] = ds_day_budget['temp_tendency_in_mld']  + pme_cor*rho0*Cp + adv_cor1*rho0*Cp + adv_cor2*rho0*Cp + eta_smoother_cor*rho0*Cp

    return(ds_day_budget)


In [ ]:
def mlt_budget_fixedh_allterms(ds_day_budget):
    """
    Like mlt_budget_fixedh in process_online_budget_year.sub, but instead of summing each
    group in bud_var_grps into a single variable, keep every individual term separate.
    """

    # Extract each individual variable, dividing by rho0*Cp to convert to degC/second:
    mlt_budget = (ds_day_budget[bud_tendency]/rho0/Cp).rename('fixedh_tendency').to_dataset()
    for grp in bud_var_grps.keys():
        for var in bud_var_grps[grp]:
            mlt_budget[var] = ds_day_budget[var]/rho0/Cp

    # Compute residual for check:
    mlt_budget['residual'] = mlt_budget['fixedh_tendency'].copy(deep=True)
    for var in list(mlt_budget.data_vars):
        if var not in ['fixedh_tendency','residual']:
            mlt_budget['residual'] -= mlt_budget[var]

    return(mlt_budget)

def compute_tendency_entrainment(mlt_budget,mlt_snap):

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)


## Compute the November 2019 monthly-mean budget (all terms)

In [ ]:
ds_day_budget = compute_corrections(ds_day_budget)

mlt_budget_allterms_daily = mlt_budget_fixedh_allterms(ds_day_budget)
mlt_budget_allterms_daily = compute_tendency_entrainment(mlt_budget_allterms_daily, ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_allterms_daily = mlt_budget_allterms_daily.load()

# Monthly mean (just November 2019):
mlt_budget_allterms_online_nov2019 = mlt_budget_allterms_daily.mean('time')


## Plot: all individual terms, online budget, November 2019

Same style as the first spatial plot in the "Plots comparing offline and daily budgets" section of `Offline_Online_Budget_Comparison.ipynb` (online row only, full field), but every term in `bud_var_grps` is plotted separately instead of grouped.

In [ ]:
# Top row: main budget terms (the dominant term from each group), all on a +-5 degC/month colorbar.
main_vars = ['mlt_tendency','entrainment','temp_advection_in_mld_cor','temp_vdiffuse_sbc_in_mld','sw_heat_in_mld']
main_labels = ['Tendency','Entrainment','Advection (resolved)','Surface fluxes','Shortwave penetration']
main_clim = 5

# Remaining individual terms, each with its own (smaller) colorbar limit - edit values here as needed:
other_labels = {
    'temp_vdiffuse_diff_cbt_in_mld':'Vertical diffusion',
    'temp_nonlocal_KPP_in_mld':'KPP nonlocal flux',
    'temp_submeso_in_mld':'Submesoscale advection',
    'neutral_diffusion_in_mld_temp':'Neutral diffusion (horizontal)',
    'neutral_gm_in_mld_temp':'GM advection',
    'temp_vdiffuse_k33_in_mld':'Neutral diffusion (vertical)',
    'temp_rivermix_in_mld':'River mixing',
    'frazil_3d_in_mld':'Frazil ice',
    'sfc_hflux_pme_in_mld_cor':'P-E+R heat flux',
    'temp_eta_smooth_in_mld_cor':'SSH-smoother',
}
other_clims = {var:0.5 for var in other_labels.keys()}
other_clims['temp_vdiffuse_diff_cbt_in_mld'] = 5
other_clims['temp_nonlocal_KPP_in_mld'] = 5

unit_conv = 86400*30.5 # degC/sec -> degC/month
fname = 'MLT_budget_Nov2019_Online_AllTerms.png'

nrows, ncols = 3, 5
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(17,8.5), layout='constrained')

vars_plot = main_vars + list(other_labels.keys())
labels_plot = main_labels + list(other_labels.values())
clims_plot = [main_clim]*len(main_vars) + [other_clims[var] for var in other_labels.keys()]

for i, (var, label, clim) in enumerate(zip(vars_plot, labels_plot, clims_plot)):
    ax = axs.reshape(-1)[i]
    (mlt_budget_allterms_online_nov2019[var]*unit_conv).plot(
        ax=ax, cmap='RdBu_r', vmin=-clim, vmax=clim, extend='both',
        cbar_kwargs={'label':label + ' ($^\circ$C/month)','shrink':0.8,'location':'top','ticks':[-clim,0,clim]}
    )
    ax.set_facecolor([0.5,0.5,0.5])
    ax.set_title('')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.text(reg[0]+1, reg[3]-1, f'({string.ascii_lowercase[i]})', verticalalignment='top', color='w')

fig.suptitle('Online MLT budget, November 2019 - all terms')
plt.savefig(fname, dpi=200, bbox_inches='tight')


# Monthly offline budget: all individual terms (November 2019)

Same idea as above, but using the monthly offline binning method from `process_offline_monthly_budget_year.sub` instead of the online diagnostics.

## Load monthly offline data

Only the standard monthly diagnostics (`ocean_month.nc`) and the 3D monthly budget (`ocean_budget_month_3d.nc`) are needed (monthly snapshots aren't used by the offline binning - `mlt_tendency`/`entrainment` are instead copied from the online calculation above, as in `Offline_Online_Budget_Comparison.ipynb`).

In [ ]:
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

# Standard monthly diagnostics:
ds_mon = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = ds_mon.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon.time.values]})
ds_mon.average_DT.data = ds_mon.average_DT*np.timedelta64(1,'D')
ds_mon = ds_mon.sel(time=month_str)

# 3D budget data:
ds_mon_budget_3d = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon_budget_3d = ds_mon_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget_3d.time.values]})
ds_mon_budget_3d.average_DT.data = ds_mon_budget_3d.average_DT*np.timedelta64(1,'D')
ds_mon_budget_3d = ds_mon_budget_3d.sel(time=month_str)

# Add sfc_hflux_pme into the 3D budget ds from the standard monthly diagnostics file:
ds_mon_budget_3d['sfc_hflux_pme'] = ds_mon['sfc_hflux_pme']

## Define offline binning and correction functions (all terms)

`bud_var_grps_offline` (copied from `process_offline_monthly_budget_year.sub`) defines the raw 3D variables that make up each grouped term. As for the online case, we keep every individual variable separate instead of summing each group. The offline corrections (advection, P-E+R and eta-smoother) are only computed per-group in the original script, so here they are instead added directly to the single dominant raw variable of each affected group (`temp_advection`, `sfc_hflux_pme`, `temp_eta_smooth`), renamed with a `_cor` suffix to match the online naming convention.

In [ ]:
bud_var_grps_offline = {'advection':['temp_advection','temp_submeso','temp_vdiffuse_k33','neutral_diffusion_temp','neutral_gm_temp'],
                        'vert_mixing':['temp_vdiffuse_diff_cbt','temp_nonlocal_KPP'],
                        'surface_flux':['temp_vdiffuse_sbc','frazil_3d','temp_rivermix'],
                        'sw_pen':['sw_heat']}
bud_2d_vars_offline = ['temp_eta_smooth','sfc_hflux_pme']

def mixed_layer_bin_offline(ds_budget,ds_budget_2d,mld,dzt):
    """
    Bin raw 3D (and 2D-surface layer only) time-averaged budget terms into the time-averaged mixed layer.
    """

    # Sum over mixed layer:
    ds_budget_in_mld = ds_budget.isel(st_ocean=0,drop=True).copy(deep=True)
    for var in list(ds_budget_2d.data_vars):
        ds_budget_in_mld[var] = xr.zeros_like(ds_budget['fixedh_tendency']).isel(st_ocean=0,drop=True).copy(deep=True)
    for ti in range(len(ds_budget.time)): # Loop over time
        dzt_ti = dzt.isel(time=ti).load()
        dzt_ti_bot = dzt_ti.cumsum('st_ocean')  # Depth (from free surface) of bottom of each cell
        ds_budget_ti = ds_budget.isel(time=ti).load()
        mld_ti = mld.isel(time=ti).load()
        for var in list(ds_budget.data_vars):
            ds_budget_in_mld[var][ti,:,:] = ds_budget_ti[var].where(dzt_ti_bot<mld_ti).sum('st_ocean')   # Include all of cells that lie completely in the mixed layer
            for k in range(len(ds_budget_ti.st_ocean)-1):
                ds_budget_in_mld[var][ti,:,:] += xr.where(np.logical_and(dzt_ti_bot[k,:,:]>mld_ti,dzt_ti_bot[k+1,:,:]<mld_ti),((mld_ti-dzt_ti_bot[k,:,:])/dzt_ti[k,:,:])*ds_budget_ti[var][k,:,:],0.) # Include only a fraction of cells that lie partially within the mixed layer
            ds_budget_in_mld[var][ti,:,:] = ds_budget_in_mld[var][ti,:,:]/rho0/Cp/mld_ti # Convert units from Wm-2 to degC/sec

        # Add 2D surface layer variables:
        surf_frac = xr.where(dzt_ti_bot[0,:,:]>mld_ti,mld_ti/dzt_ti[0,:,:],1.)            # In regions where the mixed layer depth is shallower than the thickness of the surface grid cell, take only that fraction from the 2D variables
        for var in list(ds_budget_2d.data_vars):
            ds_budget_in_mld[var][ti,:,:] = (surf_frac*ds_budget_2d[var][ti,:,:]/rho0/Cp/mld_ti).load()

    # Compute residual for check:
    ds_budget_in_mld['residual'] = ds_budget_in_mld['fixedh_tendency'] - ds_budget_in_mld[list(ds_budget_in_mld.data_vars)[1:]].to_array().sum('variable')

    return(ds_budget_in_mld)

def compute_tmlb(t3d,H):
    """
    Linearly interpolate t3d to H.
    """

    tmlb = np.nan*xr.zeros_like(t3d.isel(st_ocean=0)).values

    tmlb = xr.where(H<t3d.st_ocean[0],t3d.isel(st_ocean=0).values,np.nan)
    tmlb = xr.where(H>t3d.st_ocean[-1],t3d.isel(st_ocean=-1).values,tmlb)

    for s in range(len(t3d.st_ocean)-1):
        z = t3d.st_ocean[s]
        zp1 = t3d.st_ocean[s+1]
        between_pts = np.logical_and(z < H,zp1 > H)
        tz = t3d.isel(st_ocean=s)
        tzp1 = t3d.isel(st_ocean=s+1)
        tmlb = xr.where(between_pts,tzp1*(H - z)/(zp1 - z) + tz*(zp1 - H)/(zp1 - z),tmlb)
    return(tmlb)

def compute_corrections_allterms_offline(mlt_budget,ds_std,D):
    """
    Compute the offline advection, P-E+R and eta-smoother corrections, and apply them to the
    single dominant raw term of each affected group (rather than to a grouped sum).
    """

    mlt = ds_std.temp_in_mld.load()/rho0  # DegC
    pme = ds_std.pme_river.load()/rho0    # ms-1
    eta = ds_std.eta_t.load()             # m
    detadt = ds_std.eta_t_tendency.load() # ms-1
    eta_smooth = ds_std.eta_smoother.load() # ms-1
    H = ds_std.mld.load()                         # m
    dzs = ds_std.dzt.isel(st_ocean=0).load()      # m
    t3d = ds_std.temp.load()              # 3D temperature for tmlb
    if t3d.max() > 100.: t3d -= 273.15    # Convert to celsius if needed

    tmlb = compute_tmlb(t3d,H)

    surf_frac = xr.where(dzs>H,H/dzs,1.)

    pme_cor = -pme*mlt/H*surf_frac
    eta_smoother_cor = -eta_smooth*mlt/H*surf_frac
    adv_cor1 = (-detadt + pme + eta_smooth)*mlt/H*surf_frac
    adv_cor2 = detadt*tmlb*(1-H/(D+eta))/H*surf_frac

    # Apply corrections to the dominant raw term of each group:
    mlt_budget['temp_advection_cor'] = mlt_budget['temp_advection'] + adv_cor1 + adv_cor2
    mlt_budget['sfc_hflux_pme_cor'] = mlt_budget['sfc_hflux_pme'] + pme_cor
    mlt_budget['temp_eta_smooth_cor'] = mlt_budget['temp_eta_smooth'] + eta_smoother_cor
    mlt_budget = mlt_budget.drop_vars(['temp_advection','sfc_hflux_pme','temp_eta_smooth'])

    # Apply the same corrections to the total tendency:
    mlt_budget['fixedh_tendency'] = mlt_budget['fixedh_tendency'] + adv_cor1 + adv_cor2 + pme_cor + eta_smoother_cor

    return(mlt_budget)

def compute_offline_budget_ti_allterms(ds_budget_3d,ds_std,ht):
    """
    Compute the offline budget for a single time slice, keeping every individual term separate.
    """

    ds_budget_3d_reduced = ds_budget_3d['temp_tendency'].load().rename('fixedh_tendency').to_dataset()
    for grp in bud_var_grps_offline.keys():
        for var in bud_var_grps_offline[grp]:
            ds_budget_3d_reduced[var] = ds_budget_3d[var].load()

    ds_budget_3d_reduced_2d = ds_budget_3d[bud_2d_vars_offline].load()

    bud = mixed_layer_bin_offline(ds_budget_3d_reduced,ds_budget_3d_reduced_2d,ds_std.mld.load(),ds_std.dzt.load())
    bud = compute_corrections_allterms_offline(bud,ds_std,ht)

    return(bud)

## Compute the November 2019 monthly offline budget (all terms)

In [ ]:
ht = ds_grid.ht.load()

mlt_budget_allterms_monthly_offline_nov2019 = compute_offline_budget_ti_allterms(ds_mon_budget_3d, ds_mon, ht)
mlt_budget_allterms_monthly_offline_nov2019 = mlt_budget_allterms_monthly_offline_nov2019.load().isel(time=0, drop=True)

# Offline binning can't compute a tendency/entrainment term directly, so copy these from the online budget above instead:
mlt_budget_allterms_monthly_offline_nov2019['mlt_tendency'] = mlt_budget_allterms_online_nov2019['mlt_tendency']
mlt_budget_allterms_monthly_offline_nov2019['entrainment'] = mlt_budget_allterms_monthly_offline_nov2019['mlt_tendency'] - mlt_budget_allterms_monthly_offline_nov2019['fixedh_tendency']

## Plot: all individual terms, monthly offline budget, November 2019

Exactly the same terms, layout and colorbar limits as the online plot above (`vars_plot`, `labels_plot`, `clims_plot`), just mapped onto the corresponding offline variable names.

In [ ]:
# Top row: main budget terms (the dominant term from each group), all on a +-5 degC/month colorbar.
main_vars = ['mlt_tendency','entrainment','temp_advection_cor','temp_vdiffuse_sbc','sw_heat']
main_labels = ['Tendency','Entrainment','Advection (resolved)','Surface fluxes','Shortwave penetration']
main_clim = 5

# Remaining individual terms, each with its own (smaller) colorbar limit - edit values here as needed:
other_labels = {
    'temp_vdiffuse_diff_cbt':'Vertical diffusion',
    'temp_nonlocal_KPP':'KPP nonlocal flux',
    'temp_submeso':'Submesoscale advection',
    'neutral_diffusion_temp':'Neutral diffusion (horizontal)',
    'neutral_gm_temp':'GM advection',
    'temp_vdiffuse_k33':'Neutral diffusion (vertical)',
    'temp_rivermix':'River mixing',
    'frazil_3d':'Frazil ice',
    'sfc_hflux_pme_cor':'P-E+R heat flux',
    'temp_eta_smooth_cor':'SSH-smoother',
}
other_clims = {var:0.5 for var in other_labels.keys()}
other_clims['temp_vdiffuse_diff_cbt'] = 5
other_clims['temp_nonlocal_KPP'] = 5

unit_conv = 86400*30.5 # degC/sec -> degC/month
fname = 'MLT_budget_Nov2019_MonthlyOffline_AllTerms.png'

nrows, ncols = 3, 5
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(17,8.5), layout='constrained')

vars_plot = main_vars + list(other_labels.keys())
labels_plot = main_labels + list(other_labels.values())
clims_plot = [main_clim]*len(main_vars) + [other_clims[var] for var in other_labels.keys()]

for i, (var, label, clim) in enumerate(zip(vars_plot, labels_plot, clims_plot)):
    ax = axs.reshape(-1)[i]
    (mlt_budget_allterms_monthly_offline_nov2019[var]*unit_conv).plot(
        ax=ax, cmap='RdBu_r', vmin=-clim, vmax=clim, extend='both',
        cbar_kwargs={'label':label + ' ($^\circ$C/month)','shrink':0.8,'location':'top','ticks':[-clim,0,clim]}
    )
    ax.set_facecolor([0.5,0.5,0.5])
    ax.set_title('')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.text(reg[0]+1, reg[3]-1, f'({string.ascii_lowercase[i]})', verticalalignment='top', color='w')

fig.suptitle('Monthly offline MLT budget, November 2019 - all terms')
plt.savefig(fname, dpi=200, bbox_inches='tight')
